In [ ]:
import json
from openai import OpenAI

from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
client = OpenAI()

In [ ]:
# 1. Define your local tool function for weather data
def get_current_weather(location: str, unit: str = "fahrenheit") -> str:
  """Get the current weather for a given location."""
  # In a real application, you would call an external weather API here.
  loc = location.lower()
  if "tokyo" in loc:
    return json.dumps(
        {"location": "Tokyo", "temperature": "14", "unit": unit, "condition": "Sunny"}
    )
  elif "paris" in loc:
    return json.dumps(
        {
            "location": "Paris",
            "temperature": "18",
            "unit": unit,
            "condition": "Cloudy",
        }
    )
  else:
    return json.dumps(
        {
            "location": location,
            "temperature": "72",
            "unit": unit,
            "condition": "Clear",
        }
    )

In [ ]:
CUSTOMER_LOAN_DATABASE = {
    "Alice Smith": "$12,500",
    "Bob Jones": "$4,200",
    "Charlie Brown": "$0 (Fully Paid)",
    "Diana Prince": "$25,000",
    "Evan Wright": "$1,850",
    "Fiona Gallagher": "$8,900",
}

In [ ]:
# 2. Define your local tool function for get customer loan

def get_customer_loan(customer_name: str) -> str:
  """Get the outstanding loan amount for a specific customer."""
  cleaned_name = customer_name.strip().lower()

  for name, loan in CUSTOMER_LOAN_DATABASE.items():
    if cleaned_name in name.lower():
      return json.dumps({"customer": name, "loan_outstanding": loan})

  return json.dumps(
      {"error": f"Customer '{customer_name}' not found in loan records."}
  )

In [ ]:
# 3. Map function names to actual Python functions for execution
available_tools = {
    "get_current_weather": get_current_weather,
    "get_customer_loan": get_customer_loan,
}

In [ ]:
# 4. --- Define the Tool Schemas for OpenAI ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": (
                "Get the current weather in a given city or location."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": (
                            "The city and state/country, e.g. Tokyo, Japan"
                        ),
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use.",
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_loan",
            "description": (
                "Retrieve the outstanding loan amount for a customer by their"
                " name."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": (
                            "The first and last name of the customer."
                        ),
                    }
                },
                "required": ["customer_name"],
            },
        },
    },
]

In [ ]:
# 5. Define a detailed system prompt
SYSTEM_PROMPT = """You are a helpful customer service and query assistant equipped with tools.

Guidelines for Tool Usage:
1. When asked about weather or temperatures, use the `get_current_weather` tool.
2. When asked about loan balances or account status, use the `get_customer_loan` tool.
3. If a tool returns an error or indicates a customer wasn't found, politely inform the user and ask them to verify the spelling.
4. Always answer concisely and present loan/weather data clearly in human-readable language.
"""

In [ ]:
def run_conversation(query):
  # Try changing this input to test different tools!
  # e.g., "What is the weather like in Tokyo?" OR "How much does Diana Prince owe?"

  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": query},
  ]

  print(f"User: {query}\n")

  # First API call: Send user prompt and both available tools to the model
  response = client.chat.completions.create(
      model="gpt-4o-mini",  # or gpt-4o-mini / gpt-3.5-turbo
      messages=messages,
      tools=tools,
      tool_choice="auto",
  )

  response_message = response.choices[0].message

  # Check if the model decided to make a tool call
  if response_message.tool_calls:
    messages.append(response_message)

    # Execute each tool call requested by the model
    for tool_call in response_message.tool_calls:
      function_name = tool_call.function.name
      function_to_call = available_tools[function_name]

      # Parse arguments provided by the model
      function_args = json.loads(tool_call.function.arguments)
      print(f"-> Model selected tool: '{function_name}'")
      print(f"-> Arguments passed: {function_args}")

      # Execute the Python function dynamically
      tool_output = function_to_call(**function_args)
      print(f"-> Tool output: {tool_output}\n")

      # Append the tool's response back into the conversation history
      messages.append(
          {
              "tool_call_id": tool_call.id,
              "role": "tool",
              "name": function_name,
              "content": tool_output,
          }
      )

    # Second API call: Send the tool results back to the model for the final answer
    second_response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages
    )

    print(f"Assistant: {second_response.choices[0].message.content}")
  else:
    print(f"Assistant: {response_message.content}")

In [ ]:
run_conversation("What is the weather in Paris")

In [ ]:
run_conversation("What is the outstanding loan for Alice Smith")

In [ ]:
run_conversation("What is the outstanding loan for Fiona Gallagher")

In [ ]:
run_conversation("What is the outstanding loan for Charlie Brown")